# 02 · Árboles, Random Forest y ensembles

Los árboles capturan relaciones no lineales, interacciones y reglas fáciles de explicar. Los ensembles reducen sus debilidades combinando múltiples modelos.

## Objetivos
- Entender splits, impureza Gini y entropía.
- Controlar overfitting con profundidad, tamaño de hoja y poda.
- Comparar bagging, Random Forest, Extra Trees y boosting.
- Analizar feature importance y sus limitaciones.
- Usar validación cruzada para elegir complejidad.


## 1. Intuición matemática
En clasificación, un árbol busca cortes que reduzcan impureza. Gini para un nodo es:

$$Gini=1-\sum_k p_k^2$$

Un árbol profundo puede memorizar ruido. Los hiperparámetros `max_depth`, `min_samples_leaf`, `min_samples_split` y `ccp_alpha` actúan como regularizadores estructurales.


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, f1_score, classification_report
SEED=42
X,y=load_wine(return_X_y=True,as_frame=True)
Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.25,random_state=SEED,stratify=y)

## 2. Un árbol interpretable
Primero entrenamos un árbol pequeño para poder visualizar reglas. En producción, la interpretabilidad local de un árbol puede ser valiosa cuando una decisión debe auditarse.


In [ ]:
tree=DecisionTreeClassifier(max_depth=3,min_samples_leaf=4,random_state=SEED).fit(Xtr,ytr)
print(classification_report(yte,tree.predict(Xte),digits=3))
plt.figure(figsize=(18,8)); plot_tree(tree,feature_names=X.columns,class_names=['0','1','2'],filled=True,rounded=True); plt.show()

## 3. Complejidad y overfitting
Un árbol sin restricciones suele alcanzar accuracy de entrenamiento muy alta. Lo importante es la generalización.


In [ ]:
rows=[]
for d in list(range(1,16))+[None]:
    m=DecisionTreeClassifier(max_depth=d,random_state=SEED).fit(Xtr,ytr)
    rows.append([str(d),m.score(Xtr,ytr),m.score(Xte,yte)])
df=pd.DataFrame(rows,columns=['depth','train','test']); display(df)
plt.plot(range(len(df)),df.train,label='train'); plt.plot(range(len(df)),df.test,label='test'); plt.xticks(range(len(df)),df.depth,rotation=45); plt.legend(); plt.show()

## 4. Bagging vs Random Forest vs Extra Trees
- **Bagging:** entrena modelos en muestras bootstrap y promedia/vota.
- **Random Forest:** además selecciona subconjuntos aleatorios de features por split; reduce correlación entre árboles.
- **Extra Trees:** aleatoriza aún más los puntos de corte; puede bajar variance y ser muy rápido.

Los árboles no requieren escalado y manejan interacciones naturalmente.


In [ ]:
models={
 'Tree':DecisionTreeClassifier(max_depth=5,random_state=SEED),
 'RandomForest':RandomForestClassifier(n_estimators=400,max_features='sqrt',random_state=SEED,n_jobs=-1),
 'ExtraTrees':ExtraTreesClassifier(n_estimators=400,random_state=SEED,n_jobs=-1),
 'GradientBoosting':GradientBoostingClassifier(random_state=SEED),
 'HistGradientBoosting':HistGradientBoostingClassifier(random_state=SEED)
}
rows=[]
for name,m in models.items():
    cv=cross_val_score(m,Xtr,ytr,cv=5,scoring='f1_macro')
    m.fit(Xtr,ytr); p=m.predict(Xte)
    rows.append([name,cv.mean(),accuracy_score(yte,p),f1_score(yte,p,average='macro')])
pd.DataFrame(rows,columns=['modelo','CV F1','test accuracy','test F1']).sort_values('test F1',ascending=False).round(3)

## 5. Boosting
Boosting construye modelos secuencialmente para corregir errores previos. `GradientBoosting` ajusta nuevos árboles al gradiente de la pérdida. Implementaciones modernas como XGBoost, LightGBM y CatBoost añaden regularización, optimizaciones y manejo eficiente de grandes datasets; tendrán un lab propio.


## 6. Importancia de variables: cuidado con la interpretación
`feature_importances_` basada en impureza puede favorecer variables continuas o de alta cardinalidad. La **permutation importance** mide cuánto cae la métrica al permutar una feature y suele ser una comprobación más útil. Ninguna de las dos implica causalidad.


In [ ]:
rf=models['RandomForest']
imp=pd.Series(rf.feature_importances_,index=X.columns).sort_values(ascending=False)
perm=permutation_importance(rf,Xte,yte,n_repeats=20,random_state=SEED,n_jobs=-1)
perm_s=pd.Series(perm.importances_mean,index=X.columns).sort_values(ascending=False)
fig,ax=plt.subplots(1,2,figsize=(13,5)); imp.head(10).sort_values().plot.barh(ax=ax[0],title='Impurity importance'); perm_s.head(10).sort_values().plot.barh(ax=ax[1],title='Permutation importance'); plt.tight_layout(); plt.show()

## Casos de uso
- scoring tabular con relaciones no lineales;
- detección de riesgo y fraude;
- predicción de churn;
- priorización de casos;
- variables mixtas donde un baseline lineal queda corto;
- modelos tabulares con necesidad de explicabilidad razonable.

## Errores comunes
- dejar árboles crecer sin límite;
- usar feature importance como causalidad;
- ajustar hiperparámetros mirando el test;
- ignorar class imbalance;
- comparar modelos sin CV;
- usar demasiados árboles pequeños en boosting con learning rate alto.

## Ejercicios
1. Optimiza `max_depth` y `min_samples_leaf` con `GridSearchCV`.
2. Compara Gini vs entropy.
3. Implementa cost-complexity pruning con `ccp_alpha`.
4. Crea un dataset con 2 features y dibuja las fronteras de decisión.
5. Compara Random Forest con XGBoost/LightGBM en el lab avanzado.
6. Investiga SHAP TreeExplainer y compáralo con permutation importance.
